## Data Loading and SQLite Database Setup

In [8]:
import pandas as pd
import sqlite3

# Load the dataset from a CSV file into a pandas DataFrame
df = pd.read_csv("/content/TASK 1.csv")

# Create an in-memory SQLite database connection
conn = sqlite3.connect(":memory:")
# Write the DataFrame to a SQL table named 'orders'
df.to_sql("orders", conn, index=False, if_exists="replace")

print("Database ready — table 'orders' loaded with", len(df), "rows")

Database ready — table 'orders' loaded with 1200 rows


## Orders with High Total Price

In [9]:
# Query to find orders with a TotalPrice greater than 3000
# Results are ordered by TotalPrice in descending order
result = pd.read_sql_query("""
    SELECT OrderID, Product, TotalPrice, OrderStatus
    FROM orders
    WHERE TotalPrice > 3000
    ORDER BY TotalPrice DESC
""", conn)
# Display the query results
print(result)

      OrderID  Product  TotalPrice OrderStatus
0   ORD200789   Tablet     3456.40   Delivered
1   ORD201122  Monitor     3390.95    Returned
2   ORD200632   Laptop     3390.80   Delivered
3   ORD200469    Chair     3384.90   Cancelled
4   ORD200328   Tablet     3370.20   Cancelled
5   ORD200107  Printer     3353.75     Shipped
6   ORD200326   Laptop     3352.40    Returned
7   ORD201065  Printer     3334.00   Delivered
8   ORD201031    Phone     3322.55     Pending
9   ORD200463   Laptop     3313.90     Shipped
10  ORD200361  Printer     3299.25   Delivered
11  ORD200367   Laptop     3293.85     Pending
12  ORD200837    Chair     3277.75   Delivered
13  ORD200527    Chair     3267.35   Cancelled
14  ORD200768   Tablet     3267.30   Cancelled
15  ORD200889  Monitor     3253.60   Cancelled
16  ORD200540   Laptop     3243.25     Pending
17  ORD200802    Chair     3223.20   Cancelled
18  ORD200957  Monitor     3219.45    Returned
19  ORD200086  Printer     3215.15   Cancelled
20  ORD200221

## Product-wise Order Count

In [10]:
# Query to calculate the total number of orders for each product
# Results are grouped by Product and ordered by Total_Orders in descending order
result = pd.read_sql_query("""
    SELECT Product,
           COUNT(*) AS Total_Orders
    FROM orders
    GROUP BY Product
    ORDER BY Total_Orders DESC
""", conn)
# Display the query results
print(result)

   Product  Total_Orders
0  Printer           181
1   Tablet           179
2    Chair           178
3   Laptop           173
4     Desk           170
5  Monitor           163
6    Phone           156


## Product-wise Revenue and Average Order Value

In [11]:
# Query to calculate total orders, total revenue, and average order value per product
# Results are grouped by Product and ordered by Total_Revenue in descending order
result = pd.read_sql_query("""
    SELECT Product,
           COUNT(*) AS Total_Orders,
           SUM(TotalPrice) AS Total_Revenue,
           AVG(TotalPrice) AS Avg_Order_Value
    FROM orders
    GROUP BY Product
    ORDER BY Total_Revenue DESC
""", conn)
# Display the query results
print(result)

   Product  Total_Orders  Total_Revenue  Avg_Order_Value
0    Chair           178      195620.11      1098.989382
1  Printer           181      195612.61      1080.732652
2   Laptop           173      192126.56      1110.558150
3   Tablet           179      186568.95      1042.284637
4  Monitor           163      175651.41      1077.616012
5     Desk           170      167459.93       985.058412
6    Phone           156      151722.39       972.579423


## Product-wise Cancelled Orders

In [12]:
# Query to count the number of cancelled orders for each product
# Filters for orders where OrderStatus is 'Cancelled'
# Results are grouped by Product and ordered by Cancelled_Orders in descending order
result = pd.read_sql_query("""
    SELECT Product,
           COUNT(*) AS Cancelled_Orders
    FROM orders
    WHERE OrderStatus = 'Cancelled'
    GROUP BY Product
    ORDER BY Cancelled_Orders DESC
""", conn)
# Display the query results
print(result)

   Product  Cancelled_Orders
0    Chair                45
1  Printer                35
2  Monitor                35
3   Laptop                35
4     Desk                35
5   Tablet                34
6    Phone                31


## Payment Method Analysis

In [13]:
# Query to analyze payment methods, counting total orders and summing total revenue
# Filters for payment methods with more than 240 orders
# Results are grouped by PaymentMethod and ordered by Total_Revenue in descending order
result = pd.read_sql_query("""
    SELECT PaymentMethod,
           COUNT(*) AS Total_Orders,
           SUM(TotalPrice) AS Total_Revenue
    FROM orders
    GROUP BY PaymentMethod
    HAVING COUNT(*) > 240
    ORDER BY Total_Revenue DESC
""", conn)
# Display the query results
print(result)

  PaymentMethod  Total_Orders  Total_Revenue
0        Online           258      262442.94
1          Cash           246      259786.29


## Monthly Order and Revenue Trends

In [14]:
# Query to analyze monthly order count and total monthly revenue
# Extracts the year and month from the Date column
# Results are grouped by month and ordered chronologically
result = pd.read_sql_query("""
    SELECT SUBSTR(Date, 1, 7) AS Month,
           COUNT(*) AS Orders,
           ROUND(SUM(TotalPrice), 2) AS Monthly_Revenue
    FROM orders
    GROUP BY Month
    ORDER BY Month ASC
""", conn)
# Display the query results
print(result)

      Month  Orders  Monthly_Revenue
0   2023-01      47         56685.75
1   2023-02      37         40117.66
2   2023-03      43         48609.37
3   2023-04      31         27751.71
4   2023-05      49         63836.84
5   2023-06      45         49500.19
6   2023-07      44         42820.66
7   2023-08      51         54352.14
8   2023-09      29         29526.67
9   2023-10      47         52607.85
10  2023-11      41         43079.67
11  2023-12      46         43754.73
12  2024-01      32         38528.08
13  2024-02      32         36909.57
14  2024-03      36         36030.90
15  2024-04      50         49613.14
16  2024-05      34         27909.11
17  2024-06      53         68068.54
18  2024-07      43         42963.98
19  2024-08      28         31991.07
20  2024-09      44         39794.98
21  2024-10      31         37226.97
22  2024-11      35         32413.76
23  2024-12      41         38785.77
24  2025-01      27         29099.40
25  2025-02      37         35317.55
2

## Summary

In [15]:
print("""
=== SQL ANALYSIS SUMMARY ===

1. PROBLEM: Extract actionable business insights using structured SQL queries.

2. METHODOLOGY: SQLite in-memory database, SELECT/WHERE/GROUP BY/HAVING/ORDER BY.

3. KEY FINDINGS:
   - 34 orders exceed $3,000 — concentrated in Tablet, Laptop, Chair categories.
   - Chair leads revenue ($195,620) but has the most cancellations (45) — investigate.
   - Online payment dominates with 258 orders and $262,442 revenue.
   - June 2024 peak: 53 orders, $68,068 — highest single month in the dataset.
   - May 2023 and June 2024 are consistent high-revenue months worth targeting.

4. RECOMMENDATIONS:
   - Address Chair cancellations — highest revenue product with highest cancel rate is a risk.
   - Prioritize Online payment infrastructure — it drives the most volume and revenue.
   - Plan promotions around May/June — data shows these months consistently outperform.
""")


=== SQL ANALYSIS SUMMARY ===

1. PROBLEM: Extract actionable business insights using structured SQL queries.

2. METHODOLOGY: SQLite in-memory database, SELECT/WHERE/GROUP BY/HAVING/ORDER BY.

3. KEY FINDINGS:
   - 34 orders exceed $3,000 — concentrated in Tablet, Laptop, Chair categories.
   - Chair leads revenue ($195,620) but has the most cancellations (45) — investigate.
   - Online payment dominates with 258 orders and $262,442 revenue.
   - June 2024 peak: 53 orders, $68,068 — highest single month in the dataset.
   - May 2023 and June 2024 are consistent high-revenue months worth targeting.

4. RECOMMENDATIONS:
   - Address Chair cancellations — highest revenue product with highest cancel rate is a risk.
   - Prioritize Online payment infrastructure — it drives the most volume and revenue.
   - Plan promotions around May/June — data shows these months consistently outperform.

